In [255]:
%reset -f

In [256]:
import os
import pandas as pd
import xlwings as xw
import numpy as np

In [ ]:
# Enter Client-Specific Parameters: 

client = "Quanata"  #key word at beginning of file name
client_ISL = 75000
threshold = float(50000) #only change the number

#tab names
tab_experience = "Exp MED-1"
tab_claims = "Claims Exceeding De-Identifi-1"

#rows above the column names for LC
header_claims = 7

#file path - link FOLDER
base_path = r'C:\Users\Morgan.Godley\OneDrive - Sequoia\VS Studio Python Folder\Large Claimants Data'

In [258]:
# Set path, client name, find client files
client_files = [f for f in os.listdir(base_path) if client.lower() in f.lower() and f.endswith(".xls")]

# Print whether client files were found
if not client_files:
    print(f"No .xls files found for client: {client}")
else:
    print(f"Found file(s) for {client}: {client_files}")

    # Manipulate the First File found; use xlwings (xw) package to manipulate .xls documents
    client_file1 = client_files[0]
    client_final_xls_path = os.path.join(base_path, client_file1) # Reset path to First File
    client_final_xls = xw.Book(client_final_xls_path)             # Open file with xw package
    
    # Pull Experience & Large Claims YTD tabs
    for tab in client_final_xls.sheets:
        tab_name = tab.name
        # Pull Experience tab into DF df_exp
        if tab_name.startswith(tab_experience):
            df_exp = tab.used_range.options(pd.DataFrame, header=0, index=False).value
            print(f"\nExperience Sheet: {tab_name}, saved as DataFrame df_exp")
        # Pull Large Claims tab into DF df_lc
        elif tab_name.startswith(tab_claims):
            df_lc = tab.used_range.options(pd.DataFrame, header=0, index=False).value
            print(f"\nLarge Claims Sheet: {tab_name}, saved as DataFrame df_lc")

    client_final_xls.close()

Found file(s) for Quanata: ['Quanata Monthly Exp. Rpt_12_2025.xls']

Experience Sheet: Exp MED-1, saved as DataFrame df_exp

Large Claims Sheet: Claims Exceeding De-Identifi-1, saved as DataFrame df_lc


<h1>Large Claims YTD

In [259]:
#df_lc.head(15)

Prepare Large Claims DF

In [260]:
# Remove Cigna's header
df_lc_cleaned = df_lc.iloc[header_claims:].reset_index(drop=True)         # Drop first 10 rows
df_lc_cleaned.columns = df_lc_cleaned.iloc[0]                  # Row 11 becomes headers
df_lc_cleaned = df_lc_cleaned.iloc[1:].reset_index(drop=True)  # Drop the header row

# Convert $ columns to numeric
cols_convert = ["DRUG CLAIMS", "PAID CLAIMS", "CLAIMANT TOTAL"]

for col in cols_convert:
    df_lc_cleaned[col] = (
        df_lc_cleaned[col]
        .astype(str)                             # convert to string
        #.str.replace(r"[\$,]", "", regex=True)  # remove $ and commas
        .replace("None", np.nan)                 # convert 'None' strings to np.nan
        .astype(float)                           # convert to float
    )

# Dominate Diagnosis per Member
dominant_icd = (df_lc_cleaned.loc
                # Find the largest CLAIMANT TOTAL for each MEMBER ID
                [df_lc_cleaned.groupby("MEMBER ID")["CLAIMANT TOTAL"].idxmax()]
                # Select corresponding ICD DESCRIPTION for max CLAIMANT TOTAL
                .set_index("MEMBER ID")["ICD DESCRIPTION"])
# Replace all ICD DESCRIPTION values by MEMBER ID with dominant ICD code
df_lc_cleaned["ICD DESCRIPTION"] = df_lc_cleaned["MEMBER ID"].map(dominant_icd)

In [261]:
df_lc_cleaned.head(15)

,FUNDING TYPE,RATING TYPE,MEMBER ID,REL,ICD CODE,ICD DESCRIPTION,ICD VERSION,DRUG CLAIMS,PAID CLAIMS,CLAIMANT TOTAL
0,2.0,R,66666932612365,SB,K60329,"ANAL FISTULA, COMPLEX, UNSPECIFIED",10.0,10870.09,15264.61,26134.70
1,None,None,None,None,None,NaN,None,NaN,NaN,NaN
2,2.0,R,66666502612365,SB,K3530,"ACUTE APPENDICITIS WITH LOC PERITONITIS, W/O P...",10.0,8091.31,19080.86,27172.17
3,None,None,None,None,None,NaN,None,NaN,NaN,NaN
4,2.0,R,66666022612365,SB,S83511A,SPRAIN OF ANTERIOR CRUCIATE LIGAMENT OF RIGHT ...,10.0,5.54,38098.38,38103.92
5,None,None,None,None,None,NaN,None,NaN,NaN,NaN
6,1.0,M,66666356512365,SB,*,UNSPECIFIED,*,0.00,387280.26,387280.26
7,2.0,R,66666356512365,SB,C719,UNSPECIFIED,10.0,3495.15,37675.07,41170.22
8,,,MEMBER ID Total,,,,,3495.15,424955.33,428450.48
9,None,None,None,None,None,NaN,None,NaN,NaN,NaN


Subset Large Claims

In [262]:
# Keep Relevant Columns: MEMBER ID, REL, ICD DESCRIPTION, DRUG CLAIMS, PAID CLAIMS
df_lc_cleaned = df_lc_cleaned.copy()
if "MEMBER STATUS" not in df_lc_cleaned.columns:
    df_lc_cleaned["MEMBER STATUS"] = np.nan  

df_lc_cleaned_subset = df_lc_cleaned[["MEMBER ID", "MEMBER STATUS", "REL", "ICD DESCRIPTION",
                                      "DRUG CLAIMS", "PAID CLAIMS", "CLAIMANT TOTAL"
                                      ]].copy()

# Drop rows with "MEMBER ID Total" in MEMBER ID column
df_lc_cleaned_subset = df_lc_cleaned_subset[df_lc_cleaned_subset["MEMBER ID"] != "MEMBER ID Total"]
df_lc_cleaned_subset = df_lc_cleaned_subset.dropna(subset=["MEMBER ID"]).reset_index(drop=True)

# Group on MEMBER ID, sum numeric columns, for non-numeric columns, keep the first value
df_lc_cleaned_subset = df_lc_cleaned_subset.groupby("MEMBER ID").agg(
    {"MEMBER STATUS": "first",
     "REL": "first",
     "ICD DESCRIPTION": "first",
     "DRUG CLAIMS": "sum",
     "PAID CLAIMS": "sum",
     "CLAIMANT TOTAL": "sum"}).reset_index()

#Re-order columns
df_lc_cleaned_subset = df_lc_cleaned_subset[
    ["MEMBER ID", "REL", "MEMBER STATUS", "ICD DESCRIPTION", 
     "PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL"]]

# Sort CLAIMANT TOTAL in descending order
df_lc_cleaned_subset = df_lc_cleaned_subset.sort_values("CLAIMANT TOTAL", ascending=False).reset_index(drop=True)


cols_num = ["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL", "ISL"]

df_lc_cleaned_subset[cols_num] = (df_lc_cleaned_subset[cols_num]
                                  .apply(pd.to_numeric, errors="coerce") #make sure they're numbers
                                  .round(2))                             #2 decimal places

df_lc_cleaned_subset[["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL", "ISL"]].dtypes

In [263]:
# ISL - medical claims-only
df_lc_cleaned_subset["ISL"] = np.where(
    df_lc_cleaned_subset["PAID CLAIMS"] > client_ISL,
    df_lc_cleaned_subset["PAID CLAIMS"] - client_ISL,
    np.nan
)

# Round numeric columns to 2 decimal places
df_lc_cleaned_subset[["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL", "ISL"]] = (
    df_lc_cleaned_subset[["PAID CLAIMS", "DRUG CLAIMS", "CLAIMANT TOTAL", "ISL"]].round(2))

# Keep Claims only >= threshold
df_lc_cleaned_subset = df_lc_cleaned_subset[df_lc_cleaned_subset["CLAIMANT TOTAL"] >= threshold].copy()

In [264]:
#df_lc_cleaned_subset.head(15)

<h2>Large Claims YTD - Final Output

In [265]:
df_lc_ytd_final = df_lc_cleaned_subset.drop(columns=["MEMBER ID"])

In [266]:
df_lc_ytd_final

,REL,MEMBER STATUS,ICD DESCRIPTION,PAID CLAIMS,DRUG CLAIMS,CLAIMANT TOTAL,ISL
0,SB,NaN,UNSPECIFIED,424955.33,3495.15,428450.48,349955.33
1,SP,NaN,UNSPECIFIED,216791.36,437.78,217229.14,141791.36
2,SP,NaN,UNSPECIFIED,210379.39,292.42,210671.81,135379.39
3,CH,NaN,UNSPECIFIED,179402.76,167.58,179570.34,104402.76
4,SP,NaN,"INTERVERTEBRAL DISC DISORDERS W RADICULOPATHY,...",140426.10,3773.13,144199.23,65426.10
5,SB,NaN,MULTIPLE SCLEROSIS,139844.05,104.24,139948.29,64844.05
6,SP,NaN,MYASTHENIA GRAVIS WITHOUT (ACUTE) EXACERBATION,76437.03,58527.00,134964.03,1437.03
7,SB,NaN,OTHER ACUTE POSTPROCEDURAL PAIN,15740.94,68980.00,84720.94,NaN
8,SB,NaN,"CHRONIC SINUSITIS, UNSPECIFIED",79517.13,435.92,79953.05,4517.13
9,CH,NaN,"SCHIZOPHRENIA, UNSPECIFIED",72137.03,5833.43,77970.46,NaN


In [268]:

file_name = f"{client} - Large Claims.xlsx"
output_path = os.path.join(base_path, file_name)

# Overwrites file if it already exists
df_lc_ytd_final.to_excel(output_path, index=False)

